# Lab 14: Segmentation

**Module 14** | Read `notes/14-segmentation-unet-maskrcnn.pdf` before running this notebook.

Run a pre-trained DeepLabV3 on a sample image and visualize semantic segmentation masks.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## Semantic segmentation with DeepLabV3

DeepLabV3 combines atrous convolutions and a lightweight decoder head to assign a class label to every pixel. The ResNet-50 variant pre-trained on COCO outputs 21 Pascal-VOC-style classes (background + 20 objects).

Unlike detection (boxes), segmentation produces a dense label map. This lab runs inference, prints a per-class pixel histogram, identifies the dominant classes in the scene, colorizes the mask, and shows input versus segmentation side by side.


### Load model and sample image

Torchvision weights include the exact preprocessing (resize + ImageNet normalization) the network expects. Category names in the weight metadata follow the VOC label set.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from torchvision.models.segmentation import (
    DeepLabV3_ResNet50_Weights,
    deeplabv3_resnet50,
)
from runtime_env import ensure_sample_images

ROOT = Path("..").resolve()
IMAGE_DIR = ensure_sample_images()

weights = DeepLabV3_ResNet50_Weights.DEFAULT
model = deeplabv3_resnet50(weights=weights)
model.eval().to(device)
preprocess = weights.transforms()
class_names = weights.meta["categories"]

image_paths = sorted(IMAGE_DIR.glob("*.jpg"))
if not image_paths:
    raise FileNotFoundError(
        f"No JPG in {IMAGE_DIR}. Check your network connection or run: python download_datasets.py"
    )
img_path = image_paths[0]
original = Image.open(img_path).convert("RGB")

print(f"Model: DeepLabV3 ResNet-50 (VOC classes)")
print(f"Number of classes: {len(class_names)}")
print(f"Input image: {img_path.name} ({original.size[0]}x{original.size[1]})")
print()
print("VOC class labels:")
for idx, name in enumerate(class_names):
    print(f"  {idx:2d}: {name}")


### Run inference

The output `out["out"]` has shape `[1, num_classes, H, W]`. Taking `argmax` over the class dimension yields the per-pixel label. We then count how many pixels belong to each class to understand scene composition.


In [ ]:
batch = preprocess(original).unsqueeze(0).to(device)
with torch.no_grad():
    out = model(batch)["out"]

mask = out.argmax(dim=1).squeeze(0).cpu().numpy()
unique_ids = np.unique(mask)

print(f"Segmentation map shape: {mask.shape}")
print(f"Unique class ids in scene: {unique_ids.tolist()}")
print()

counts = np.bincount(mask.flatten(), minlength=len(class_names))
total_pixels = mask.size

print("Class histogram (pixel counts):")
print("=" * 55)
print(f"{'ID':>3}  {'Class':<18}  {'Pixels':>10}  {'Percent':>8}")
print("-" * 55)
for cls_id in range(len(class_names)):
    if counts[cls_id] > 0:
        pct = 100.0 * counts[cls_id] / total_pixels
        print(f"{cls_id:3d}  {class_names[cls_id]:<18}  {counts[cls_id]:>10,}  {pct:7.2f}%")
print("-" * 55)
print(f"{'':3}  {'TOTAL':<18}  {total_pixels:>10,}  {'100.00%':>8}")


### Top classes and detected VOC labels

Sorting classes by pixel count reveals what dominates the scene (often background, sky, or large surfaces). The detected set lists which of the 20 VOC object classes (excluding background) appear with at least one pixel.


In [ ]:
nonzero = [(i, counts[i]) for i in range(len(class_names)) if counts[i] > 0]
ranked = sorted(nonzero, key=lambda x: x[1], reverse=True)

print("Top classes in scene (by pixel count):")
for rank, (cls_id, count) in enumerate(ranked[:8], start=1):
    pct = 100.0 * count / total_pixels
    print(f"  {rank}. {class_names[cls_id]:<18} {count:>8,} px ({pct:.2f}%)")

detected_objects = [
    class_names[i] for i in unique_ids if i != 0 and class_names[i] != "__background__"
]
print()
print(f"VOC object classes detected (excluding background): {len(detected_objects)}")
if detected_objects:
    print("  " + ", ".join(detected_objects))
else:
    print("  (none, scene may be mostly background or unlabeled regions)")


### Colorize and compare

We map each class id to a distinct RGB color (deterministic via a fixed seed) and show input versus colored mask side by side. Background stays black so foreground objects stand out.


In [ ]:
rng = np.random.default_rng(42)
num_classes = out.shape[1]
palette = rng.integers(0, 255, size=(num_classes, 3), dtype=np.uint8)
palette[0] = [0, 0, 0]

colored = palette[mask]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original)
axes[0].set_title(f"Input: {img_path.name}")
axes[0].axis("off")

axes[1].imshow(colored)
axes[1].set_title("DeepLabV3 segmentation (colored classes)")
axes[1].axis("off")

plt.tight_layout()
plt.show()


### Evaluation

Verify that the histogram sums to the total pixel count, that detected VOC classes match what you see in the side-by-side plot, and that dominant classes make sense for the image content. Segmentation errors often appear at object boundaries or on small instances that occupy few pixels.


In [ ]:
print("Segmentation evaluation:")
print(f"  Image: {img_path.name}")
print(f"  Map resolution: {mask.shape[0]}x{mask.shape[1]}")
print(f"  Classes with pixels: {len(nonzero)}")
print(f"  VOC objects detected: {detected_objects}")
print(f"  Dominant class: {class_names[ranked[0][0]]} ({100.0 * ranked[0][1] / total_pixels:.1f}% of pixels)")
